# Modelos de Predicción

Este notebook entrena y evalúa modelos de machine learning para predecir enfermedades cardíacas.

## Objetivos:
1. Preprocesamiento de datos
2. Entrenamiento de modelos
3. Evaluación y comparación
4. Optimización de hiperparámetros


In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías importadas")


In [ ]:
# Cargar y preparar datos
df = pd.read_csv('../data/CVD_cleaned.csv')

# Codificar variables categóricas
le = LabelEncoder()
df['Heart_Disease_encoded'] = le.fit_transform(df['Heart_Disease'])
df['Exercise_encoded'] = le.fit_transform(df['Exercise'].astype(str))

# Seleccionar features
features = ['BMI', 'Exercise_encoded', 'Age_Category']
X = pd.get_dummies(df[features + ['Age_Category']], drop_first=True)
y = df['Heart_Disease_encoded']

print(f"Features: {X.shape[1]}")
print(f"Muestras: {X.shape[0]}")


In [ ]:
# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} muestras")
print(f"Test: {len(X_test)} muestras")


In [ ]:
# Entrenar modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)

# Predecir
y_pred = modelo.predict(X_test)
y_pred_proba = modelo.predict_proba(X_test)[:, 1]

# Métricas
print("Métricas del modelo:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")


In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Guardar modelo
joblib.dump(modelo, 'modelo_heart_disease.pkl')
print("\n✓ Modelo guardado en: modelo_heart_disease.pkl")
